# P18: how much can we trust the museum B numbers?

Three questions the ADRs left open:

1. **Why does every model score higher on museum B than on validation?** ADR 0011 and 0012 guessed that museum B, which holds only
   named artists, is easier than validation, two thirds of which is "Unknown Artist". Never checked.
2. **Do the models learn the painter instead of the genre?** ADR 0003 built museum B with unseen artists for that reason, and
   proposed to "repeat with a few seeds and report the spread". Museum B is one draw of 44 artists; here museum A is split by
   artist many times instead.
3. **Which gaps between models are real?** So far each museum B score had a +/- 2.9 point interval computed alone. Two models
   scored on the same 600 paintings can be compared **paired**, painting by painting, which is much more sensitive.

**The rules, fixed before any result** (a scratch run, unsaved, checked the code):

- Nothing is trained to be chosen here; no model changes. Museum B predictions are computed for the models already scored on it,
  plus two references never scored before (the frozen ResNet-18 head and the dense baseline, both re-run on this computer in
  P13 and P14), **only for the paired tests**: no decision depends on them.
- **Question 2:** museum A (2,400 paintings) is split 5 times into 5 folds, by artist (no named artist in two folds; each
  "Unknown Artist" painting on its own) and at random (stratified by genre). Each fold is predicted by a model trained on the 4
  others for a **fixed number of epochs** (the best epochs of P06 and P14: 20 for the ResNet head, 5 for the feature network; the
  forest has none), so the held-out fold chooses nothing. The artist effect is the accuracy on named-artist paintings when their
  artist was in training minus when it was not.
- **Question 3:** six paired comparisons fixed in advance, each with the exact McNemar test (on the paintings where exactly one of
  the two models is right) and a 95% bootstrap interval of the difference (10,000 resamplings of the 600 paintings); the p-values
  are corrected for six tests (Holm).
- Results go to `results/museum_b_tests.csv` and `results/grouped_cv.csv`. Running the notebook again gives the same numbers.

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold

from deep_learning_project_waerniers_lamy_mahieddine.vision.utils import (
    DATA_DIR, RESULTS_DIR, build_dense, build_feature_mlp, build_resnet_head, final_model, fit, load_data, load_handcrafted,
    load_resnet_data, load_result, predict_logits, quantize_int8, split_images, summary)

table = pd.read_csv(DATA_DIR / "split.csv")
runs = summary()


def saved_model(name, build):
    """The seed-0 model of the earliest saved run with this name, with its weights."""
    stem = runs[runs["name"] == name].sort_values("date")["file"].iloc[0]
    model = build(load_result(stem)["config"])
    model.load_state_dict(torch.load(RESULTS_DIR / f"{stem}.pt"))
    return model.eval()


def predict(model, X):
    with torch.no_grad():
        return model(X).argmax(dim=1)


# Predictions of every model on validation and museum B (a few minutes: features and images)
val_images, y_val = split_images("val")
test_images, y_test = split_images("test")
preds = {}
final = final_model()
preds["final ResNet-18 (fine-tuned)"] = (predict_logits(final, val_images).argmax(1), predict_logits(final, test_images).argmax(1))
train_images, _ = split_images("train")
calibration = [train_images[i] for i in torch.randperm(len(train_images), generator=torch.Generator().manual_seed(0))[:200].tolist()]
int8 = quantize_int8(final, calibration)  # the int8 model of P16, same calibration
preds["final ResNet-18, int8"] = (predict_logits(int8, val_images).argmax(1), predict_logits(int8, test_images).argmax(1))
resnet = load_resnet_data({"finetune": "head"}, splits=("train", "val", "test"))
head = saved_model("resnet18_head_rerun", build_resnet_head)
preds["ResNet-18 frozen"] = (predict(head, resnet["X_val"]), predict(head, resnet["X_test"]))
mobile = load_resnet_data({"model": "mobilenet_v3_large"}, splits=("val", "test"))
mobile_head = saved_model("mobilenet_v3_large_head", build_resnet_head)
preds["MobileNetV3-Large frozen"] = (predict(mobile_head, mobile["X_val"]), predict(mobile_head, mobile["X_test"]))
handcrafted = load_handcrafted(splits=("train", "val", "test"))
feature_net = saved_model("features_mlp256_drop_wd", build_feature_mlp)
preds["feed-forward on handcrafted features"] = (predict(feature_net, handcrafted["X_val"]), predict(feature_net, handcrafted["X_test"]))
raw = np.load(DATA_DIR / "handcrafted.npz")["X"]
split = table["split"].to_numpy()
forest = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(raw[split == "train"], handcrafted["y_train"].numpy())
preds["random forest (P11)"] = (torch.tensor(forest.predict(raw[split == "val"])), torch.tensor(forest.predict(raw[split == "test"])))
pixels = load_data(32, splits=("train", "val", "test"))
for name, label in [("dense_flip", "dense, mirrored copies (final dense)"), ("dense_baseline_rerun", "dense baseline (P02, re-run)")]:
    dense = saved_model(name, build_dense)
    preds[label] = (predict(dense, pixels["X_val"]), predict(dense, pixels["X_test"]))

for label, (pv, pt) in preds.items():
    print(f"{label:38s} validation {(pv == y_val).float().mean():.1%} | museum B {(pt == y_test).float().mean():.1%}")

final ResNet-18 (fine-tuned)           validation 83.3% | museum B 86.0%
final ResNet-18, int8                  validation 83.3% | museum B 86.2%
ResNet-18 frozen                       validation 80.2% | museum B 84.5%
MobileNetV3-Large frozen               validation 82.7% | museum B 84.8%
feed-forward on handcrafted features   validation 60.4% | museum B 62.7%
random forest (P11)                    validation 60.4% | museum B 58.2%
dense, mirrored copies (final dense)   validation 43.5% | museum B 41.0%
dense baseline (P02, re-run)           validation 39.0% | museum B 40.8%


## 1. Is museum B easier? Validation split by kind of artist

Validation holds 161 paintings by named artists (all of whom also have paintings in training) and 319 by "Unknown Artist".
Museum B holds only named artists, none of them seen in training. 95% intervals by bootstrap.

In [2]:
named = (table.loc[table["split"] == "val", "artist"] != "Unknown Artist").to_numpy()
rng = np.random.default_rng(0)


def interval(correct, n_boot=10000):
    """Mean and 95% bootstrap interval of a 0/1 array."""
    correct = np.asarray(correct, dtype=float)
    means = correct[rng.integers(0, len(correct), (n_boot, len(correct)))].mean(axis=1)
    return correct.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


rows = []
for label, (pv, pt) in preds.items():
    right_val, right_test = (pv == y_val).numpy(), (pt == y_test).numpy()
    row = {"model": label}
    for part, correct in [("val, named artists (161)", right_val[named]), ("val, Unknown Artist (319)", right_val[~named]),
                          ("museum B (600)", right_test)]:
        m, lo, hi = interval(correct)
        row[part] = f"{m:.1%} [{lo:.1%}, {hi:.1%}]"
        row[part + " value"] = m
    rows.append(row)
by_artist = pd.DataFrame(rows).set_index("model")
by_artist["named - unknown (points)"] = 100 * (by_artist["val, named artists (161) value"] - by_artist["val, Unknown Artist (319) value"])
by_artist["museum B - val named (points)"] = 100 * (by_artist["museum B (600) value"] - by_artist["val, named artists (161) value"])
by_artist[[c for c in by_artist.columns if not c.endswith(" value")]].round(1)

,"val, named artists (161)","val, Unknown Artist (319)",museum B (600),named - unknown (points),museum B - val named (points)
model,,,,,
final ResNet-18 (fine-tuned),"85.7% [80.1%, 90.7%]","82.1% [77.7%, 86.2%]","86.0% [83.2%, 88.7%]",3.6,0.3
"final ResNet-18, int8","85.7% [80.1%, 91.3%]","82.1% [77.7%, 86.2%]","86.2% [83.3%, 88.8%]",3.6,0.5
ResNet-18 frozen,"83.2% [77.6%, 88.8%]","78.7% [74.0%, 83.1%]","84.5% [81.5%, 87.3%]",4.5,1.3
MobileNetV3-Large frozen,"86.3% [80.7%, 91.3%]","80.9% [76.5%, 85.3%]","84.8% [81.8%, 87.7%]",5.5,-1.5
feed-forward on handcrafted features,"64.6% [57.1%, 72.0%]","58.3% [53.0%, 63.6%]","62.7% [58.7%, 66.5%]",6.3,-1.9
random forest (P11),"66.5% [59.0%, 73.3%]","57.4% [52.0%, 62.7%]","58.2% [54.3%, 62.0%]",9.1,-8.3
"dense, mirrored copies (final dense)","42.9% [35.4%, 50.3%]","43.9% [38.2%, 49.2%]","41.0% [37.0%, 45.0%]",-1.0,-1.9
"dense baseline (P02, re-run)","42.2% [34.8%, 50.3%]","37.3% [32.0%, 42.6%]","40.8% [36.8%, 44.8%]",4.9,-1.4


**What we see.**

- **The "museum B is easier" puzzle is explained by the kind of artist.** For every network, the validation paintings by named artists score about the same as museum B (differences from -1.9 to +1.3 points; the final model: 85.7% against 86.0%), and the "Unknown Artist" paintings score lower (final model 82.1%, 3.6 points below its named artists). Validation is two thirds "Unknown Artist", museum B has none: that alone gives the 1.5 to 5 points by which every model scored higher on museum B (ADR 0011, 0012).
- Why "Unknown Artist" paintings are harder is not tested here: they are a catch-all of many painters and periods (ADR 0003).
- **The forest is the exception:** 66.5% on validation's named artists, whose painters are all in training, but 58.2% on museum B's unseen painters, 8.3 points lower. It is the only model that seems to lean on the painter. The intervals on 161 paintings are wide (about +/- 7 points), so section 2 measures it properly.
- The two dense networks are near 40% everywhere: too weak for the kind of artist to matter.

## 2. Do the models learn the painter? Museum A split by artist, many times

Three models fast enough to be trained 50 times: the frozen ResNet-18 with its linear head, the feed-forward network on
handcrafted features (P14) and the random forest (P11). The fine-tuned final model takes about 10 minutes per training, so it is
not repeated here. About 2 minutes.

In [3]:
order = np.concatenate([np.where(table["split"] == "train")[0], np.where(table["split"] == "val")[0]])  # the order of the features
A = table.iloc[order].reset_index(drop=True)
y_A = torch.tensor(A["label_idx"].to_numpy())
named_A = (A["artist"] != "Unknown Artist").to_numpy()
groups = np.where(named_A, A["artist"], A["id"].astype(str))  # each "Unknown Artist" painting is its own group
X_resnet = torch.cat([resnet["X_train"], resnet["X_val"]])
X_raw = raw[order]


def resnet_head(train, test):
    torch.manual_seed(0)
    model = build_resnet_head({"finetune": "head"})
    fit(model, {"X_train": X_resnet[train], "y_train": y_A[train], "X_val": X_resnet[test], "y_val": y_A[test]},
        epochs=20, seed=0, verbose=False, keep_best=False)
    return predict(model, X_resnet[test]).numpy()


def feature_network(train, test):
    mean, std = X_raw[train].mean(axis=0), X_raw[train].std(axis=0) + 1e-8  # standardised with the training folds only
    X = torch.tensor((X_raw - mean) / std, dtype=torch.float32)
    torch.manual_seed(0)
    model = build_feature_mlp({"n_features": X.shape[1], "hidden": 256, "dropout": 0.5})
    fit(model, {"X_train": X[train], "y_train": y_A[train], "X_val": X[test], "y_val": y_A[test]},
        epochs=5, weight_decay=1e-2, seed=0, verbose=False, keep_best=False)
    return predict(model, X[test]).numpy()


def random_forest(train, test):
    return RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(X_raw[train], y_A[train].numpy()).predict(X_raw[test])


cv_rows = []
for label, train_and_predict in [("ResNet-18 frozen", resnet_head), ("feed-forward on handcrafted features", feature_network),
                                 ("random forest (P11)", random_forest)]:
    for repeat in range(5):
        for kind, splitter in [("random", StratifiedKFold(5, shuffle=True, random_state=repeat)),
                               ("by artist", GroupKFold(5, shuffle=True, random_state=repeat))]:
            predicted = np.empty(len(y_A), dtype=int)
            fold_named = []
            for train, test in splitter.split(np.zeros(len(y_A)), y_A.numpy(), groups):
                predicted[test] = train_and_predict(train, test)
                in_fold = test[named_A[test]]
                fold_named.append((predicted[in_fold] == y_A[in_fold].numpy()).mean())
            correct = predicted == y_A.numpy()
            cv_rows.append({"model": label, "repeat": repeat, "folds": kind, "named artists": correct[named_A].mean(),
                            "Unknown Artist": correct[~named_A].mean(), "all": correct.mean(),
                            "named, spread across folds (std)": np.std(fold_named, ddof=1)})
cv = pd.DataFrame(cv_rows)
cv.to_csv(RESULTS_DIR / "grouped_cv.csv", index=False)

pivot = cv.pivot_table(index=["model", "repeat"], columns="folds", values="named artists")
pivot["artist effect (points)"] = 100 * (pivot["random"] - pivot["by artist"])
effect = pivot.groupby("model").agg(**{"named, artist seen": ("random", "mean"), "named, artist unseen": ("by artist", "mean"),
                                       "artist effect, mean (points)": ("artist effect (points)", "mean"),
                                       "smallest": ("artist effect (points)", "min"), "largest": ("artist effect (points)", "max")})
spread = cv[cv["folds"] == "by artist"].groupby("model")["named, spread across folds (std)"].mean()
effect["unseen artists: std across folds (points)"] = 100 * spread
print("written: results/grouped_cv.csv")
effect.round(3)

written: results/grouped_cv.csv


,"named, artist seen","named, artist unseen","artist effect, mean (points)",smallest,largest,unseen artists: std across folds (points)
model,,,,,,
ResNet-18 frozen,0.825,0.810,1.538,-0.504,3.783,4.830
feed-forward on handcrafted features,0.600,0.580,1.992,-1.009,4.540,5.484
random forest (P11),0.590,0.557,3.354,1.387,4.792,4.777


**What we see.**

- **The artist shortcut is small.** On named-artist paintings, a model whose training saw the painter is right more often than one whose training did not, by **1.5 points for the frozen ResNet-18** (from -0.5 to +3.8 over the 5 repeats), **2.0 for the feed-forward network** on handcrafted features (-1.0 to +4.5) and **3.4 for the forest** (+1.4 to +4.8, positive in every repeat). The forest, built on colour and edge statistics, is the model that leans most on the painter, as section 1 suggested; the networks barely do. ADR 0003 feared the artist could give the genre away (it predicts it for 70% of the named paintings); for these models it is worth 1.5 to 3.4 points.
- **Which artists land in the held-out part matters more than the shortcut:** across the 25 artist-grouped folds, the accuracy on unseen named artists has a standard deviation of 4.8 to 5.5 points (about 160 paintings per fold). Museum B is one such draw, with 600 paintings: its 86.0% could have been a few points lower or higher with another choice of 44 artists.
- The fine-tuned final model was not repeated here (10 minutes per training); the frozen ResNet-18 uses the same pretrained features.